# [Exercise] Lab 1 — Neural Architecture Search on HW-NAS-Bench

## Search space, estimation strategy, and simple search algorithms

In this lab, we will work with a tabular benchmark: a CSV file extracted from [HW-NAS-Bench](https://openreview.net/pdf?id=_0kaDkv3dVf$0), which contains, for each architecture in the [NAS-Bench-201](https://openreview.net/forum?id=HJxyZkBKDr$0) search space, the accuracy obtained after training and several hardware metrics (latency, energy, ...) measured on different devices.

Having a tabular benchmark means that "training and evaluating" a network costs a table lookup: we can therefore compare search algorithms in a few seconds, repeating experiments across many seeds, which would be impossible if we had to actually train every network.

Lab objectives

1. Understand how a cell-based search space is structured and how an architecture is encoded.
2. Separate the three components of a NAS system: search space, search strategy, performance estimation strategy.
3. Implement and compare Grid Search, Random Search, and Bayesian Optimization.
4. Extend the search to the hardware-aware case (latency constraints, Pareto front).
5. (Bonus) Take a first look at evolutionary algorithms and Reinforcement Learning.


## 0. Setup

In [ ]:
import itertools
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

from utils import OPS, EDGES, load_benchmark, draw_cell, Oracle, make_gp, plot_regret

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

## 1. Data loading

The benchmark is downloaded automatically. `load_benchmark` returns a table with **one row per architecture** (15625 in total) and always the same columns, whatever dataset and device you choose:

| column | meaning |
| --- | --- |
| `arch_str` | architecture string in NAS-Bench-201 format |
| `op_e1_0` … `op_e3_2` | operation on each edge (`op_eJ_I` = edge from node I to node J) |
| `ops` | the same 6 operations as a tuple, handy for the search algorithms |
| `acc` | **validation** accuracy (%) after 200 epochs: this is what we optimize |
| `test_acc` | test accuracy (%): use it only to report the final result |
| `lat`, `energy` | latency (ms) and energy (mJ) on the chosen device |
| `cost` | training time (s), used to simulate the search cost |
| `flops`, `params` | FLOPs (M) and parameters (M) |

You can change `DATASET` and `DEVICE`. Not all devices provide every metric: energy is available only for `edgegpu`, `eyeriss` and `fpga` (otherwise the column is empty).

In [ ]:
DATA_URL = f"hw_nasbench201.csv.gz"   # or a local path

DATASET = "cifar100"   # cifar10 | cifar100 | imagenet16
DEVICE = "raspi4"      # edgegpu | raspi4 | edgetpu | pixel3 | eyeriss | fpga

df = load_benchmark(DATA_URL, dataset=DATASET, device=DEVICE)

N_ARCH = len(df)
ARCH2IDX = {ops: i for i, ops in enumerate(df["ops"])}   # tuple of ops -> row index
BEST_ACC = df["acc"].max()                               # best accuracy in the whole space
df.head()

## 2. Benchmark exploration

Before searching, let's look at the data: how are the accuracies distributed? How much does an accurate network cost (in latency)?

In [ ]:
print(df[["acc", "test_acc", "lat", "energy", "cost", "flops", "params"]].describe().round(3))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(df["acc"], bins=80, color="tab:blue")
ax[0].set(xlabel="Accuracy (%)", ylabel="# architectures", title="Accuracy distribution")
ax[1].scatter(df["lat"], df["acc"], s=3, alpha=0.3)
ax[1].set(xlabel=f"Latency on {DEVICE} (ms)", ylabel="Accuracy (%)", title="Accuracy vs latency")
plt.tight_layout(); plt.show()

best = df.loc[df["acc"].idxmax()]
print("Best architecture:", best["arch_str"], f"→ {best['acc']:.2f}%")
print("99th percentile of accuracy:", np.percentile(df["acc"], 99).round(2))

### ❓ Questions — NAS vs Hyperparameter tuning

Some questions?

## 3. The search space: cell-based design

In NAS-Bench-201, the network consists of a fixed skeleton (stem, 3 stages of stacked cells, reduction blocks between stages, classifier). Search concerns only the cell, which is then repeated.

The cell is a DAG with 4 nodes (0 = input, 3 = output). Each pair of nodes $i<j$ is connected by an edge, for a total of 6 edges, and on each edge one operation is chosen from:

| operation | meaning |
| --- | --- |
| `none` | absent edge (zero) |
| `skip_connect` | identity |
| `nor_conv_1x1` | ReLU-Conv1x1-BN |
| `nor_conv_3x3` | ReLU-Conv3x3-BN |
| `avg_pool_3x3` | 3x3 average pooling |

Each node sums the contributions of incoming edges.

In [ ]:
draw_cell(best, title=f"Best architecture ({best['acc']:.2f}%)")

### Exercise 1 — Search space analysis

**1a.** Calculate analytically the cardinality of the search space (hint: use $n_{OPS}^{n_{EDGES}}$) and compare it with the number of rows in the CSV.

**1b.** Some cells do not connect the input to the output (all paths contain a `none` edge): whatever the rest of the network is, the cell produces zero. Complete the `is_connected` function and count how many there are. What accuracy do they have in the benchmark?

<details>
<summary>💡 Hint</summary>

First, build a list of 6 booleans, one per edge, that is `True` if the edge exists (i.e. its operation is not `none`). For example, for the operations `("nor_conv_3x3", "none", "skip_connect", "none", "none", "avg_pool_3x3")` the list is `[True, False, True, False, False, True]`.

Then work node by node, in order, and ask: *can node k receive anything from the input?*

- Node 1 is reachable if its only incoming edge (from node 0) exists.
- Node 2 is reachable if the edge 0→2 exists, **or** if the edge 1→2 exists **and** node 1 is reachable.
- Apply the same idea to node 3: the cell is connected if node 3 is reachable.

Remember the edge order in `EDGES`: `(1,0), (2,0), (2,1), (3,0), (3,1), (3,2)`.
</details>

In [ ]:
# 1a
#TODO: calculate the theoretical cardinality of the search space
space_size = ...
print("Theoretical cardinality:", space_size, "| row in the CSV:", N_ARCH)

In [ ]:
# 1b
def is_connected(row):
    # return True if the cell is connected (input → output)
    ops =[]
    for dst, src in EDGES:
        ops.append(row[f"op_e{dst}_{src}"])
    #TODO: check if the cell is connected
    return ...

df["connected"] = [is_connected(row) for _, row in df.iterrows()]
print("Unconnected cells:", (~df["connected"]).sum())
print(df.groupby("connected")["acc"].describe().round(2))
print(df[~df["connected"]][["arch_str", "acc"]].head())
draw_cell(df[~df["connected"]].iloc[0], title="Example of unconnected cell")

### ❓ Questions — Cell-based design
1. What are the advantages of searching for a repeating cell instead of an entire network? And the limitations?

## 4. Components of NAS

```
 ┌────────────────┐   Architecture  ┌───────────────────────────────┐
 │ Search strategy│ ───────────────▶│ Performance estimation        │
 │  (GS, RS, BO,  │                 │ strategy (qui: lookup nel     │
 │   RL, EA, ...) │ ◀───────────────│ benchmark = complete training)│
 └────────────────┘    Evaluation   └───────────────────────────────┘
          │ samples from
          ▼
 ┌────────────────┐
 │  Search space  │  (cell NB201: 5^6 architectures)
 └────────────────┘
```

In our case, the estimation strategy is an oracle: it returns the final accuracy as if we had trained the network. To make the comparison fair, the oracle:

* counts queries and enforces a budget (maximum number of evaluable architectures);
* accumulates the simulated cost (training time, if present in the CSV);
* can add noise to simulate a low-fidelity estimation (e.g., few epochs).

The oracle is implemented in utils.py. To use it, these methods are sufficient:

|  |  |
| --- | --- |
| Oracle(df, budget, noise_std=0.0, seed=0) | creates the oracle with a query budget |
| oracle.query(idx) | accuracy of the architecture in row idx of df |
| oracle.query_ops(ops) | same, given the tuple of 6 operations |
| oracle.remaining() | queries still available |
| oracle.best() | (ops, accuracy) of the best found so far |
| oracle.incumbent_curve() | best accuracy after each query |

In [ ]:
o = Oracle(df, budget=3)
print(o.query(0), o.query(1), o.query(0), "| queries used:", o.n_queries)   # re-query is free
print("Best so far:", o.best())

## 5. Grid Search

Grid Search systematically enumerates the Cartesian product of all choices. In NB201 the "full grid" coincides with the whole search space (15625 points): with a budget of 100 evaluations we can see less than 1% of it.

Two typical variants when the budget is limited:
- **truncated full grid**: enumerate in lexicographic order and stop when the budget runs out;
- **reduced grid**: choose a subset of operations for each edge (e.g. 2 choices → $2^6=64$ points).

### EXERCISE 2
Implement `grid_search`. Use `itertools.product`; skip combinations that are not in the dataset; stop when the budget is exhausted.

<details>
<summary>💡 Hint</summary>

**1. Enumerating the grid.** `ops_per_edge` is a list of 6 lists, one per edge, with the operations allowed on that edge. `itertools.product(*ops_per_edge)` yields every possible combination as a tuple of 6 operations, in lexicographic order. The `*` unpacks the list, so each inner list becomes a separate argument. A small example with 2 edges:

    list(itertools.product(["a", "b"], ["x", "y"]))
    # [('a', 'x'), ('a', 'y'), ('b', 'x'), ('b', 'y')]

Notice that the **last** edge changes fastest and the **first** one slowest.

**2. Skipping missing architectures.** Each tuple has the same format as the keys of `ARCH2IDX`. If a tuple is not a key (`ops not in ARCH2IDX`), that architecture is not in the CSV: skip it with `continue`. Skipping does not cost any budget.

**3. Respecting the budget.** Before each query, check `oracle.remaining()`: if it is `0`, stop the loop with `break`. Calling `query` with no budget left raises `BudgetExhausted`.

**4. Evaluating.** Use `oracle.query_ops(ops)`. You don't need to store the result yourself: the oracle keeps the history, and you can read it later with `oracle.best()` and `oracle.incumbent_curve()`.

**5. Returning.** At the end, `return oracle`.

The whole function body is a single `for` loop of about 6 lines.
</details>

In [ ]:
def grid_search(oracle, ops_per_edge=None):
    # ops_per_edge: list of 6 lists of allowed operations for each arc
#               (None = all operations on all arcs)
    if ops_per_edge is None:
        ops_per_edge = [OPS] * 6
    #TODO: enumerate the grid and query the oracle
    return oracle


BUDGET = 100
#TODO: run the grid search with the full grid and with a reduced grid  and print the best accuracy and regret

### ❓ Questions — Grid Search
1. Why does the "truncated complete" grid search depend so heavily on the order of operations? Which edges are actually explored with 100 queries?
2. Does the reduced grid perform well or poorly? What does this depend on? Who selected the grid operations (and based on what prior knowledge)?

## 6. Random Search

Random Search samples architectures uniformly (without repetition) until the budget is exhausted. It is surprisingly competitive and it is the mandatory baseline for any NAS work.

### EXERCISE 3
Implement `random_search` by sampling indices without replacement.

<details>
<summary>💡 Hint</summary>

**1. Working with indices.** Each architecture is a row of `df`, so it is identified by an integer index from `0` to `N_ARCH - 1`. Unlike grid search, you don't need to build tuples of operations: you can sample indices directly and query them with `oracle.query(idx)`.

**2. Sampling without replacement.** The random generator `rng` is already created for you from the `seed`. The simplest way to get indices without repetitions is a random permutation:

    rng.permutation(5)
    # e.g. array([3, 0, 4, 1, 2])

It returns all the numbers from `0` to `n - 1` exactly once, in random order. Iterating over it means sampling uniformly without replacement.

**3. Respecting the budget.** As in grid search, check `oracle.remaining()` before each query and stop with `break` when it reaches `0`.

**4. Returning.** At the end, `return oracle`.

**Why the seed?** Using `rng` (and never `np.random` directly) makes each run reproducible: the same `seed` gives the same sequence of architectures. In Section 8 we will repeat the search with different seeds to average over the randomness.
</details>

In [ ]:
def random_search(oracle, seed=0):
    rng = np.random.default_rng(seed)
    #TODO: sample indices without replacement and query the oracle
    return oracle


rs = random_search(Oracle(df, BUDGET), seed=0)
#TODO: print the best accuracy and regret

### ❓ Question
Let $p$ be the fraction of architectures in the top 1% ($p=0.01$). What is the probability that Random Search finds at least one of them within $n$ evaluations? Calculate this for $n=100$ and verify it empirically using the following cell.

In [ ]:
thr = np.percentile(df["acc"], 99)
hits = [max(v for _, v in random_search(Oracle(df, BUDGET), seed=s).history) >= thr
        for s in range(200)]
print(f"Empirical: {np.mean(hits):.3f}   Theoretical: {1 - 0.99**BUDGET:.3f}")

## 7. Bayesian Optimization

BO builds a surrogate model (here a Gaussian Process) that predicts the accuracy of architectures not yet evaluated, along with an uncertainty estimate. An acquisition function decides which architecture to evaluate next, balancing exploitation (high mean) and exploration (high uncertainty).

Encoding. The GP works on vectors: we encode each architecture in one-hot (6 edges x 5 operations = 30 dimensions).

Expected Improvement. Given the best observed value $f^*$, mean $\mu(x)$, and standard deviation $\sigma(x)$:

$$z = \frac{\mu(x) - f^* - \xi}{\sigma(x)}, \qquad \mathrm{EI}(x) = (\mu(x) - f^* - \xi)\,\Phi(z) + \sigma(x)\,\varphi(z)$$

where $\Phi$ and $\varphi$ are the CDF and PDF of the standard normal distribution ($\mathrm{EI}=0$ if $\sigma=0$).

Since the search space is small, we can evaluate the acquisition function on all architectures not yet queried and take the maximum.

In [ ]:
def encode_onehot(ops):
    v = np.zeros(len(EDGES) * len(OPS))
    for e, op in enumerate(ops):
        v[e * len(OPS) + OPS.index(op)] = 1.0
    return v

X_ALL = np.stack(df["ops"].map(encode_onehot).to_numpy())
print("Coding matrix:", X_ALL.shape)

# make_gp() (Gaussian Process with Matern kernel) is defined in utils.py

### EXERCISE 4 — Expected Improvement

Implement `expected_improvement` using the formula above.

<details>
<summary>💡 Hint</summary>

**1. Work on whole arrays.** `mu` and `sigma` are NumPy arrays with one value per candidate architecture. Write the formula with array operations (`+`, `-`, `*`, `/`), with no `for` loops: NumPy applies them element by element.

**2. The pieces of the formula.** Compute them in this order:
- the improvement: `imp = mu - best_f - xi`
- the standardized value: `z = imp / sigma`
- the EI: `imp * Φ(z) + sigma * φ(z)`

For Φ and φ use `norm.cdf(z)` and `norm.pdf(z)`. `norm` is already imported from `scipy.stats`.

**3. The case σ = 0.** If `sigma` is zero, `imp / sigma` divides by zero. Two steps solve it:
- before dividing, replace `sigma` with `np.maximum(sigma, 1e-12)`, a tiny positive number;
- at the end, set EI to `0` where `sigma` was (almost) zero, with `np.where(condition, ei, 0.0)`.

**4. Check your result.** The test cell should print approximately:

    [0.     0.394  0.415]

Look at the third value: a candidate with a **lower** predicted mean (85 vs 90) gets a **higher** EI, because it is much more uncertain (σ = 5). This is exploration at work.
</details>

In [ ]:
def expected_improvement(mu, sigma, best_f, xi=0.01):
    # mu, sigma: array with mean and standard dev predicted by the GP
    #TODO: implement the expected improvement formula
    return ...

# Test veloce
print(expected_improvement(np.array([90., 90., 85.]), np.array([0.0, 1.0, 5.0]), best_f=90))

### EXERCISE 5 — The BO loop

1. Evaluate `n_init` random architectures.
2. Repeat while there is budget left: fit the GP on the observed points, predict mean and standard deviation for all the architectures not yet evaluated, pick the one with the highest EI and evaluate it.

<details>
<summary>💡 Hint</summary>

**Keep two lists in sync.** Use `observed` for the row indices you have evaluated and `y` for their accuracies, in the same order. Every time you query an architecture, append to both.

**1. Initialization.** Sample `n_init` distinct indices with `rng.choice(N_ARCH, size=n_init, replace=False)`, query each of them with `oracle.query(i)`, and store the results in `y`.

**2. The loop.** Use `while oracle.remaining() > 0:` and, inside it:

- **Fit the surrogate.** Create a new GP with `gp = make_gp(seed)` and train it with `gp.fit(X, y)`. Here `X` holds the one-hot encodings of the observed architectures, `X_ALL[observed]`, and `y` must be an array, `np.array(y)`.
- **Find the candidates.** You need the indices of all architectures **not** yet evaluated. One way is a boolean mask:

      mask = np.ones(N_ARCH, dtype=bool)
      mask[observed] = False
      cand = np.flatnonzero(mask)   # indices where mask is True

- **Predict.** `mu, sigma = gp.predict(X_ALL[cand], return_std=True)` gives one mean and one standard deviation per candidate.
- **Compute EI.** Call your `expected_improvement(mu, sigma, max(y), xi)`. The `best_f` is the best accuracy observed so far.
- **Choose and evaluate.** `np.argmax(ei)` gives the **position inside `cand`**, not the row of `df`. The row index is `cand[np.argmax(ei)]`. Query it and append it to `observed` and `y`.

**3. Returning.** At the end, `return oracle`.

**Common pitfall.** If you use `np.argmax(ei)` directly as the architecture index, the code runs without errors but picks the wrong architectures. If BO performs no better than random search, check this first.
</details>

In [ ]:
def bayesian_optimization(oracle, n_init=10, xi=0.01, seed=0):
    rng = np.random.default_rng(seed)
    observed = list(rng.choice(N_ARCH, size=n_init, replace=False))
    y = [oracle.query(i) for i in observed]
    #TODO: implement the BO loop
    return oracle


#TODO: run the BO with a budget of 100 and print the best accuracy, the regret and the search time (How long does it take to fit the GP 10 times and predict 15625 points each time?)

## 8. Experimental comparison

Stochastic algorithms must be compared over **multiple seeds**. We report the **regret** (best accuracy in the benchmark − best accuracy found) as a function of the number of evaluations, on a logarithmic scale (linear close to 0, so that a zero regret remains visible).

BO is slower (one GP fit per iteration): with 10 seeds and a budget of 100 it takes about 1–3 minutes.

In [ ]:
N_RUNS = 10

def run_many(algo, n_runs=N_RUNS, budget=BUDGET, **kw):
    curves = []
    for s in range(n_runs):
        o = algo(Oracle(df, budget, seed=s), seed=s, **kw)
        c = o.incumbent_curve()
        c = np.pad(c, (0, budget - len(c)), mode="edge")
        curves.append(BEST_ACC - c)
    return np.array(curves)

results = {}
results["Grid (complete)"] = np.array([BEST_ACC - grid_search(Oracle(df, BUDGET)).incumbent_curve()])
_c = grid_search(Oracle(df, BUDGET), reduced).incumbent_curve()
results["Grid (reduced)"] = np.array([BEST_ACC - np.pad(_c, (0, BUDGET - len(_c)), mode="edge")])
t0 = time.time()
results["Random Search"] = run_many(random_search)
time_RS = time.time()-t0
print(f"RS: {time_RS:.0f}s")
t0 = time.time()
results["Bayesian Opt."] = run_many(bayesian_optimization)
print(f"BO: {time.time()-t0:.0f}s")

In [ ]:
plot_regret(results)
print(pd.DataFrame({k: {"regret finale (media)": v[:, -1].mean(),
                        "dev. std": v[:, -1].std()} for k, v in results.items()}).T.round(3))

### Questions — Comparison

1. After how many evaluations does BO consistently surpass Random Search? Why are they equivalent at the beginning?
2. How much would an experiment with a budget of 100 actually cost (in GPU-hours), if each network required ~1-2 hours of training? What does the tabular benchmark allow us to do? (Questo si può fare usando il campo `cost` del CSV, che simula il tempo di addestramento.)
3. Try changing xi in the EI (e.g., 0, 0.1, 1.0) and n_init (e.g., 2, 30). What happens?
4. Repeat the search with a noisy oracle (Oracle(..., noise_std=2.0)), which simulates a low-fidelity estimation. Which algorithm suffers the most? Why?

## 9. NAS hardware-aware

HW-NAS-Bench ci dà le metriche hardware: possiamo cercare la rete più accurata **che rispetti un vincolo di latenza**, oppure studiare il compromesso accuratezza/latenza tramite il **fronte di Pareto**.

Nota: misurare la latenza è (in questo setting) molto più economico che addestrare la rete, quindi possiamo assumere di conoscerla **gratis** per ogni architettura, prima di valutarla.

In [ ]:
def pareto_front(acc, lat):
    # Indici delle architetture non dominate (acc alta, latenza bassa)
    order = np.argsort(lat)
    front, best = [], -np.inf
    for i in order:
        if acc[i] > best:
            front.append(i); best = acc[i]
    return np.array(front)

if df["lat"].notna().any():
    pf = pareto_front(df["acc"].to_numpy(), df["lat"].to_numpy())
    plt.scatter(df["lat"], df["acc"], s=3, alpha=0.2, label="tutte")
    plt.plot(df["lat"].iloc[pf], df["acc"].iloc[pf], "r.-", label="fronte di Pareto")
    plt.xlabel("Latenza"); plt.ylabel("Accuratezza (%)"); plt.legend(); plt.show()
    print(f"Architetture sul fronte: {len(pf)}")
else:
    print("Nessuna colonna di latenza nel CSV: sezione non disponibile.")

### ✏️ ESERCIZIO 6 — Ricerca con vincolo di latenza

Scegliamo come vincolo la mediana della latenza. Modificate Random Search e BO in modo che **valutino solo architetture ammissibili** (latenza ≤ `LAT_MAX`). Confrontate il regret rispetto alla migliore architettura *ammissibile*.

Suggerimento: basta restringere l'insieme dei candidati.

In [ ]:
LAT_MAX = df["lat"].median()
FEASIBLE = np.flatnonzero(df["lat"].to_numpy() <= LAT_MAX)
BEST_FEAS = df["acc"].iloc[FEASIBLE].max()
print(f"Vincolo: latenza ≤ {LAT_MAX:.3f}  | ammissibili: {len(FEASIBLE)}  | best ammissibile: {BEST_FEAS:.2f}%")


def random_search_constrained(oracle, seed=0):
    rng = np.random.default_rng(seed)
    for idx in rng.permutation(FEASIBLE):
        if oracle.remaining() <= 0:
            break
        oracle.query(idx)
    return oracle


def bo_constrained(oracle, n_init=10, xi=0.01, seed=0):
    rng = np.random.default_rng(seed)
    observed = list(rng.choice(FEASIBLE, size=n_init, replace=False))
    y = [oracle.query(i) for i in observed]
    feas_mask = np.zeros(N_ARCH, dtype=bool); feas_mask[FEASIBLE] = True
    while oracle.remaining() > 0:
        gp = make_gp(seed); gp.fit(X_ALL[observed], np.array(y))
        mask = feas_mask.copy(); mask[observed] = False
        cand = np.flatnonzero(mask)
        mu, sigma = gp.predict(X_ALL[cand], return_std=True)
        nxt = int(cand[np.argmax(expected_improvement(mu, sigma, max(y), xi))])
        observed.append(nxt); y.append(oracle.query(nxt))
    return oracle

In [ ]:
def run_many_c(algo, n_runs=N_RUNS):
    out = []
    for s in range(n_runs):
        c = algo(Oracle(df, BUDGET, seed=s), seed=s).incumbent_curve()
        out.append(BEST_FEAS - np.pad(c, (0, BUDGET - len(c)), mode="edge"))
    return np.array(out)

res_c = {"RS vincolata": run_many_c(random_search_constrained),
         "BO vincolata": run_many_c(bo_constrained)}
plot_regret(res_c, f"Regret con vincolo latenza ≤ {LAT_MAX:.3f}")

### ❓ Domande — Hardware-aware NAS
1. La migliore architettura in assoluto è ammissibile? Quanta accuratezza "si perde" imponendo il vincolo?
2. Un'alternativa al vincolo rigido è una **funzione obiettivo scalarizzata**, ad esempio $\mathrm{acc}(x) - \lambda\,\mathrm{lat}(x)$ (approccio usato in MnasNet con una forma moltiplicativa). Quali pro e contro rispetto al vincolo?
3. *(Facoltativo)* Ripetete con la latenza di un altro dispositivo presente nel CSV. Le architetture ottime cambiano? Cosa ci dice questo sulla trasferibilità tra dispositivi?

## 10. Bonus — Uno sguardo ad altri algoritmi

### 10.1 Algoritmi evolutivi: *Regularized Evolution* (Real et al., 2019)

Si mantiene una **popolazione** di architetture. A ogni passo:
1. si estrae un **campione** casuale di `S` individui dalla popolazione (*tournament selection*);
2. il migliore del campione diventa **genitore** e viene **mutato** (si cambia l'operazione su un arco a caso);
3. il figlio viene valutato e inserito; l'individuo **più vecchio** viene rimosso (*aging*: questa è la "regolarizzazione").

In [ ]:
from collections import deque

def mutate(ops, rng):
    ops = list(ops)
    e = rng.integers(6)
    ops[e] = rng.choice([o for o in OPS if o != ops[e]])
    return tuple(ops)


def regularized_evolution(oracle, pop_size=20, sample_size=5, seed=0):
    rng = np.random.default_rng(seed)
    population = deque()
    for idx in rng.choice(N_ARCH, size=pop_size, replace=False):
        population.append((df["ops"].iat[idx], oracle.query(idx)))
    while oracle.remaining() > 0:
        sample = [population[i] for i in rng.choice(len(population), sample_size, replace=False)]
        parent = max(sample, key=lambda t: t[1])[0]
        child = mutate(parent, rng)
        while ARCH2IDX[child] in oracle.cache and oracle.remaining() > 0:
            child = mutate(child, rng)   # evitiamo di "sprecare" mutazioni già viste
        population.append((child, oracle.query_ops(child)))
        population.popleft()
    return oracle


results["Reg. Evolution"] = run_many(regularized_evolution)
plot_regret(results)

### 10.2 Reinforcement Learning (Zoph & Le, 2017) — versione minimale

Un **controller** definisce una distribuzione sulle architetture; qui, per semplicità, è una distribuzione categorica indipendente per ciascun arco, con logit $\theta_{e,o}$ (nel lavoro originale è una RNN). Si campiona un'architettura, la si valuta, e si aggiornano i logit con **REINFORCE**:
$$
\theta \leftarrow \theta + \eta\,(R - b)\,\nabla_\theta \log p_\theta(\text{arch})
$$
dove $R$ è l'accuratezza e $b$ una baseline (media mobile) che riduce la varianza.

In [ ]:
def reinforce_search(oracle, lr=0.1, baseline_decay=0.9, seed=0):
    rng = np.random.default_rng(seed)
    theta = np.zeros((6, len(OPS)))
    baseline = None
    while oracle.remaining() > 0:
        p = np.exp(theta - theta.max(1, keepdims=True)); p /= p.sum(1, keepdims=True)
        choice = [rng.choice(len(OPS), p=p[e]) for e in range(6)]
        ops = tuple(OPS[c] for c in choice)
        R = oracle.query_ops(ops)
        baseline = R if baseline is None else baseline_decay * baseline + (1 - baseline_decay) * R
        adv = (R - baseline) / 10.0           # scala del reward (punti %)
        for e, c in enumerate(choice):        # grad log-softmax = onehot - p
            grad = -p[e]; grad[c] += 1
            theta[e] += lr * adv * grad
    return oracle

results["REINFORCE"] = run_many(reinforce_search)
plot_regret(results)